In [2]:
import torch
import nltk
from transformers import LongformerModel, LongformerConfig, AutoTokenizer
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import random
from nltk.tokenize import PunktSentenceTokenizer
from sklearn.metrics import f1_score
import numpy as np
import os
from transformers import logging
import pandas as pd
import ast

shared_tokenizer = AutoTokenizer.from_pretrained('allenai/longformer-base-4096', add_prefix_space=True)
shared_tokenizer.add_special_tokens({'bos_token': '[BOS]'})

# Suppress TensorFlow warnings (not needed for this project)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.set_verbosity_error()

# print("All imports successful!")

# Download NLTK punkt tokenizer if not present
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

# print("Data Preprocessing check")
# 1. Data Preprocessing ========================================================
class WikiPreprocessor:
    def __init__(self, model_name='allenai/longformer-base-4096'):
        self.sent_tokenizer = PunktSentenceTokenizer()
        self.tokenizer = shared_tokenizer
        # self.tokenizer.add_special_tokens({'bos_token': '[BOS]'})

    def process_document(self, text):
      try:
          sentences = self.sent_tokenizer.tokenize(text)
          return [f"[BOS]{' ' + sent.strip() if sent.strip() else ''}" for sent in sentences]
      except Exception as e:
          print(f"Error processing document: {e}")
          return ["[BOS] Invalid document"]

    def create_labels(self, boundaries, num_sentences):
        """Create binary boundary labels"""
        labels = [0] * num_sentences
        for idx in boundaries:
            if idx < num_sentences:
                labels[idx] = 1
        print("labels in process_document = ",labels);
        return labels

    def sliding_window(self, sentences, max_seq_len=4096, overlap=1):
        """Handle long documents with overlapping windows"""
        windows = []
        current_window = []
        current_len = 0

        for sent in sentences:
            tokens = self.tokenizer.tokenize(sent)
            if current_len + len(tokens) > max_seq_len:
                windows.append(current_window)
                current_window = current_window[-overlap:] if overlap else []
                current_len = sum(len(self.tokenizer.tokenize(s)) for s in current_window)

            current_window.append(sent)
            current_len += len(tokens)

        if current_window:
            windows.append(current_window)

        return windows
    def __getitem__(self, idx):
        doc = self.documents[idx]['text']
        print(f"Original document: {doc}")  # Debug print
        processed = self.preprocessor.process_document(doc)
        print(f"Processed sentences: {processed}")  # Debug print
    # Rest of the method...
    def __getitem__(self, idx):
        doc = self.documents[idx]['text'].strip()
        if not doc:
            return {
                'input_ids': torch.zeros(self.max_len, dtype=torch.long),
                'attention_mask': torch.zeros(self.max_len, dtype=torch.long),
                'labels': torch.zeros(self.max_len, dtype=torch.long)
            }
    # Rest of processing...
# print("Data Preprocessing check end")

# print("Data Augumentation check")
# 2. Data Augmentation (TSSP) ==================================================
class TopicAugmentor:
    def __init__(self, p1=0.5, p2=0.5):
        self.p1 = p1  # Document augmentation probability
        self.p2 = p2  # Topic replacement probability
        self.topic_bank = []  # Storage for replacement topics

    def add_to_topic_bank(self, topics):
        """Add topics to the replacement bank"""
        self.topic_bank.extend(topics)

    def _get_random_topic(self):
        """Get random topic from bank"""
        return random.choice(self.topic_bank)

    def augment_document(self, doc, boundaries):
        """Apply TSSP augmentation"""
        topics = self._split_into_topics(doc, boundaries)

        # Store original topics for future use
        self.add_to_topic_bank(topics)

        # Shuffle topics
        if random.random() < self.p1:
            random.shuffle(topics)

        # Replace topics
        augmented_topics = []
        for topic in topics:
            if random.random() < self.p2 and self.topic_bank:
                augmented_topics.append(self._get_random_topic())
            else:
                augmented_topics.append(topic)

        # Shuffle sentences within topics
        augmented_topics = [random.sample(t, len(t)) for t in augmented_topics]

        return self._join_topics(augmented_topics)

    def _split_into_topics(self, doc, boundaries):
        topics = []
        start = 0
        for end in sorted(boundaries):
            topics.append(doc[start:end+1])
            start = end + 1
        if start < len(doc):
            topics.append(doc[start:])
        return topics

    def _join_topics(self, topics):
        return [sent for topic in topics for sent in topic]
# print("Data Augumentation check end")

# print("Model architecture check")
# 3. Model Architecture ========================================================

class TopicSegmenter(torch.nn.Module):
    def __init__(self, model_name='allenai/longformer-base-4096', dropout=0.1):
        super().__init__()
        self.tokenizer = shared_tokenizer

        # Initialize config first
        self.config = LongformerConfig.from_pretrained(model_name)

        # Then initialize model with config
        self.encoder = LongformerModel(self.config)

        # Load pretrained weights
        pretrained = LongformerModel.from_pretrained(model_name)
        self.encoder.load_state_dict(pretrained.state_dict())

        self.encoder.resize_token_embeddings(len(self.tokenizer))

        self.dropout = torch.nn.Dropout(dropout)

        self.classifier = torch.nn.Linear(self.config.hidden_size, 2)
        self.projection = torch.nn.Linear(self.config.hidden_size, 256)

    def forward(self, input_ids, attention_mask, global_attention_mask=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask
        )

        # Get BOS representations with validation
        print("BOS token ID:",self.tokenizer.bos_token_id)
        print("input_ids:", input_ids)

        bos_positions = (input_ids == self.tokenizer.bos_token_id).nonzero(as_tuple=True)
        print("BOS positions (as tuple of indices):", bos_positions)
        print("Batch indices of BOS tokens:", bos_positions[0])
        print("Sequence positions of BOS tokens:", bos_positions[1])

        # Handle case where no BOS tokens are found
        if bos_positions[0].numel() == 0:
            # Return zeros for classification and projections
            seq_len = input_ids.shape[0]
            dummy_output = torch.zeros((seq_len, self.config.hidden_size),
                                    device=input_ids.device)
            logits = torch.zeros((seq_len, 2), device=input_ids.device)
            return logits, dummy_output

        bos_embeddings = outputs.last_hidden_state[bos_positions]

        # Classification head
        logits = self.classifier(self.dropout(bos_embeddings))

        # CSSL projections

        projections = self.projection(bos_embeddings)

        print("projections shape = ",projections.shape)
        print("projections = ",projections)
        print("logits shape = ",logits.shape)
        print("logits = ",logits)

        return logits, projections
# print("Model architecture check end")

# print("Contrastive learning check")
# 4. Contrastive Learning (CSSL) ===============================================
class ContrastiveLoss:
    def __init__(self, margin=1.0, k1=1, k2=3):
        self.margin = margin
        self.k1 = k1
        self.k2 = k2
        self.cos = torch.nn.CosineSimilarity(dim=-1)

    def __call__(self, projections, boundaries):
        """
        projections: sentence embeddings [num_sentences, dim]
        boundaries: list of boundary indices
        """
        losses = []
        topics = self._split_into_topics(projections, boundaries)
        
        # Pre-compute global to local index mapping
        topic_indices = []
        current_global = 0
        for topic in topics:
            topic_indices.append((current_global, current_global + len(topic)))
            current_global += len(topic)

        for global_idx in range(len(projections)):
            # Find which topic this sentence belongs to
            topic_idx = self._find_topic(global_idx, topic_indices)
            current_topic = topics[topic_idx]
            
            # Convert global index to local topic index
            local_idx = global_idx - topic_indices[topic_idx][0]

            # Positive samples (from same topic)
            positives = self._sample_positives(local_idx, current_topic)

            # Negative samples (from other topics)
            negatives = self._sample_negatives(topics, topic_idx)

            # Calculate contrastive loss
            if len(positives) > 0 and len(negatives) > 0:
                pos_loss = torch.mean(1 - self.cos(projections[global_idx].unsqueeze(0), positives))
                neg_loss = torch.mean(torch.relu(self.cos(projections[global_idx].unsqueeze(0), negatives) - self.margin))
                losses.append(pos_loss + neg_loss)

        return torch.mean(torch.stack(losses)) if losses else torch.tensor(0.0)

    def _split_into_topics(self, projections, boundaries):
        """Split sentences into topics based on boundary indices"""
        topics = []
        start = 0
        for end in boundaries:
            topics.append(projections[start:end+1])
            start = end + 1
        if start < len(projections):
            topics.append(projections[start:])
        return topics

    def _find_topic(self, global_idx, topic_indices):
        """Find which topic contains the given global sentence index"""
        for i, (start, end) in enumerate(topic_indices):
            if start <= global_idx < end:
                return i
        return len(topic_indices) - 1

    def _sample_positives(self, local_idx, topic):
        """Sample positive pairs from the same topic using local indices"""
        if len(topic) <= 1:
            return torch.empty(0)
        candidates = [i for i in range(len(topic)) if i != local_idx]
        selected = random.sample(candidates, min(self.k1, len(candidates)))
        return torch.stack([topic[i] for i in selected])

    def _sample_negatives(self, all_topics, current_topic_idx):
        """Sample negative pairs from other topics"""
        negatives = []
        for i, topic in enumerate(all_topics):
            if i != current_topic_idx:
                negatives.extend(topic)
        if not negatives:
            return torch.empty(0)
        selected = random.sample(range(len(negatives)), min(self.k2, len(negatives)))
        return torch.stack([negatives[i] for i in selected])
# print("Contrastive learning check end")

# print("Training setup check")
# 5. Training Setup ============================================================
class TopicDataset(Dataset):
    def __init__(self, documents, tokenizer, max_len=4096):
        self.documents = documents
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.preprocessor = WikiPreprocessor()

    def __len__(self):
        return len(self.documents)

    def __getitem__(self, idx):
      doc = self.documents[idx]['text'].strip()
      boundaries = self.documents[idx]['boundaries']

      print("doc = ",doc)
      # Process document
      processed = self.preprocessor.process_document(doc)
      if not processed or all(not sent.strip() for sent in processed):
          processed = ["[BOS] Empty document"]

      labels = self.preprocessor.create_labels(boundaries, len(processed))

      print("labels in getitem", labels);
      # Tokenize with padding and truncation
      encoding = self.tokenizer(
          processed,
          max_length=self.max_len,
          padding='max_length',
          truncation=True,
          return_tensors='pt',
          is_split_into_words=True,
          add_special_tokens=True
      )

      # Create global attention mask
      print("BOS token ID:",self.tokenizer.bos_token_id)
      bos_positions = (encoding['input_ids'] == self.tokenizer.bos_token_id)
      print(idx," - ", bos_positions.sum().item())
      global_attention = torch.zeros_like(encoding['input_ids'])
      global_attention[bos_positions] = 1

      # Ensure labels match input_ids length
      # labels = labels[:self.max_len] + [0] * (self.max_len - len(labels))

      return {
          'input_ids': encoding['input_ids'].squeeze(0),
          'attention_mask': encoding['attention_mask'].squeeze(0),
          'global_attention_mask': global_attention.squeeze(0),
          'labels': torch.tensor(labels, dtype=torch.long)
      }

class Trainer:
    def __init__(self, model, train_loader, val_loader, args):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.args = args

        self.optim = AdamW(model.parameters(), lr=args['lr'])
        self.ce_loss = torch.nn.CrossEntropyLoss()
        self.cssl_loss = ContrastiveLoss(margin=1.0, k1=args.get('k1', 1), k2=args.get('k2', 3))
        self.tokenizer = shared_tokenizer
    def train_epoch(self):
      self.model.train()
      total_loss = 0.0

      print("no. of batches = ",len(self.train_loader));
      for batch in self.train_loader:
          # Skip empty batches
          if batch['input_ids'].numel() == 0:
              continue



          self.optim.zero_grad()

          # Move data to device
          input_ids = batch['input_ids'].to(self.args['device'])
          print("input_ids = " , input_ids)
          attention_mask = batch['attention_mask'].to(self.args['device'])
          global_attention_mask = batch['global_attention_mask'].to(self.args['device'])
          labels = batch['labels'].to(self.args['device'])
          print("labels = ",labels)
          # Forward pass
          outputs, projections = self.model(
              input_ids=input_ids,
              attention_mask=attention_mask,
              global_attention_mask=global_attention_mask
          )


          print("outputs = ",outputs);
          # print(projections)
          # print(labels)

          # Get number of actual BOS tokens found
          bos_positions = (input_ids == self.model.tokenizer.bos_token_id)
          print("here in this training bos_positions = ",bos_positions)
          print("here outouts",outputs);
          print("here labels",labels);
          num_bos_tokens = bos_positions.sum().item()

          # Verify alignment
          if num_bos_tokens != outputs.shape[0]:
              print(f"Mismatch: {num_bos_tokens} BOS tokens but {outputs.shape[0]} outputs")
              continue

          # Get corresponding labels (first num_bos_tokens elements)
          valid_labels = labels.view(-1)

          # Calculate losses only if we have matching pairs
          if valid_labels.numel() == outputs.shape[0]:
              print("i should be here ")
              print("outputs shape = ",outputs.shape);
              print("valid labels shape = ",valid_labels.shape);
              ce_loss = self.ce_loss(outputs, valid_labels)

              # Calculate CSSL loss
              boundaries = (valid_labels == 1).nonzero(as_tuple=True)[0].cpu().tolist()
              cssl_loss = self.cssl_loss(projections, boundaries)

              # Combined loss
              loss = self.args['alpha1'] * ce_loss + self.args['alpha2'] * cssl_loss
              loss.backward()

              torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
              self.optim.step()

              total_loss += loss.item()

      return total_loss / len(self.train_loader) if len(self.train_loader) > 0 else 0.0

    def evaluate(self):
        self.model.eval()
        total_loss = 0.0
        all_preds = []
        all_labels = []

        # print("i am here");
        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch['input_ids'].to(self.args['device'])
                attention_mask = batch['attention_mask'].to(self.args['device'])
                global_attention_mask = batch['global_attention_mask'].to(self.args['device'])
                labels = batch['labels'].to(self.args['device'])

                outputs, _ = self.model.forward(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    global_attention_mask=global_attention_mask
                )

                # Get BOS positions for valid labels
                bos_positions = (input_ids == self.tokenizer.bos_token_id).nonzero(as_tuple=True)
                print("bos_positions = ",bos_positions)

                # Check if any BOS tokens are found
                if bos_positions[0].nelement() == 0:
                    # Skip this batch if no BOS tokens are found
                    continue

                # Get valid outputs and labels using BOS positions
                valid_outputs = outputs
                print("valid_outputs = ",valid_outputs)
                valid_labels = labels.view(-1)
                print("valid_labels = ",valid_labels)
                print("outputs shape = ",outputs.shape);
                print("valid labels shape = ",valid_labels.shape);
                # Calculate loss using valid outputs and labels
                loss = self.ce_loss(outputs,valid_labels)
                total_loss += loss.item()

                preds = torch.argmax(outputs, dim=-1)
                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy().flatten())

        # print(f"all_preds shape after flattening: {all_preds.shape}")
        # print(f"all_labels shape after flattening: {all_labels.shape}")
        all_preds = np.concatenate(all_preds, axis=0)
        all_labels = np.concatenate(all_labels, axis=0)
        print(f"all_preds shape after flattening: {all_preds.shape}")
        print(f"all_labels shape after flattening: {all_labels.shape}")
        # Calculate metrics
        f1 = f1_score(all_labels, all_preds)
        return {
            'loss': total_loss / len(self.val_loader),
            'f1': f1
        }

# print("Training setup check end")


In [4]:
def load_dataset_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    dataset = []

    for _, row in df.iterrows():
        text = row['text']
        # Safely convert string like "[1, 2, 3]" to actual list
        boundaries = ast.literal_eval(row['boundaries']) if isinstance(row['boundaries'], str) else row['boundaries']
        dataset.append({'text': text, 'boundaries': boundaries})

    return dataset
def generate_results(model, data_loader, device, model_name="Combined Loss Model"):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            global_attention_mask = batch['global_attention_mask'].to(device)
            labels = batch['labels'].to(device)

            _, _, logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                global_attention_mask=global_attention_mask
            )

            preds = torch.argmax(logits, dim=-1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy().flatten())
    
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    
    # Calculate metrics
    accuracy = (all_preds == all_labels).mean()
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    
    # Print results
    print(f"\n=== {model_name} Results ===")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(5,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['No Boundary', 'Boundary'],
                yticklabels=['No Boundary', 'Boundary'])
    plt.title(f'{model_name} Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'confusion_matrix': cm
    }


In [5]:
# Usage Example =================================================================
if __name__ == "__main__":
    # # Configuration
    # try:
    #     import transformers
    #     # print(f"Transformers version: {transformers.__version__}")
    # except ImportError:
    #     print("Installing required packages...")
    #     import subprocess
    #     subprocess.run(["pip", "install", "torch", "transformers", "nltk", "scikit-learn"])

    config = {
        'model_name': 'allenai/longformer-base-4096',
        'batch_size': 4,  # Reduce from 4 to 1
        'max_seq_len': 4096,
        'lr': 5e-5,
        'epochs': 5,
        'alpha1': 0.5,
        'alpha2': 1.0,
        'k1': 1,
        'k2': 3,
        'device': 'cuda' if torch.cuda.is_available() else 'cpu'
    }

    # Initialize components
    tokenizer = shared_tokenizer
    model = TopicSegmenter(config['model_name']).to(config['device'])
    # print(model);

    # Sample data (replace with your actual dataset)

    csv_path = '/home/om/text_seg_oml/text_seg/topic_dataset.csv'
    documents = load_dataset_from_csv(csv_path)
      # Split into train/val
    train_size = max(1, int(0.8 * len(documents)))  # Ensure at least 1 sample
    train_data = documents[:train_size]
    val_data = documents[train_size:]

    print(f"Training samples: {len(train_data)}")
    print(f"Validation samples: {len(val_data)}")
    # Create datasets
    train_dataset = TopicDataset(train_data, tokenizer, config['max_seq_len'])
    val_dataset = TopicDataset(val_data, tokenizer, config['max_seq_len'])

    print(f"Train data set size : {len(train_dataset)}")
    print(f"Validation data set size: {len(val_dataset)}")


Training samples: 8
Validation samples: 3
Train data set size : 8
Validation data set size: 3


In [8]:
    # print("hi",len(train_dataset[0]['input_ids']))
    # print("how",len(train_dataset[0]['labels']))

    # print(val_dataset)
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'])

    # print(f"Train loader size: {len(train_loader)}")
    # print(f"Validation loader size: {len(val_loader)}")
    # print(train_loader)
    # print(val_loader)
    # Initialize trainer
    # sample = train_dataset[0]

    # input_ids = sample['input_ids'].unsqueeze(0).to(config['device'])
    # attention_mask = sample['attention_mask'].unsqueeze(0).to(config['device'])
    # global_attention_mask = sample['global_attention_mask'].unsqueeze(0).to(config['device'])

    # logits, projections = model(input_ids, attention_mask, global_attention_mask)

    # # # Get boundary predictions
    # predicted_boundaries = torch.argmax(logits, dim=-1).squeeze()  # (4096,)



    trainer = Trainer(model, train_loader, val_loader, config)

    # Training loop
    for epoch in range(config['epochs']):
        train_loss = trainer.train_epoch()
        val_metrics = trainer.evaluate()

        print(f"Epoch {epoch+1}/{config['epochs']}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss: {val_metrics['loss']:.4f}")

no. of batches =  2
doc =  First document. Second sentence. New topic starts here. Another sentence.
labels in process_document =  [0, 1, 1, 1]
labels in getitem [0, 1, 1, 1]
BOS token ID: 50265
0  -  4
doc =  The startup secured funding. They plan to expand. Wildlife protection is crucial. Many species are endangered.
labels in process_document =  [0, 0, 1, 0]
labels in getitem [0, 0, 1, 0]
BOS token ID: 50265
5  -  4
doc =  Astronomy explores the universe. Telescopes help see far galaxies. Traveling helps understand cultures. Food connects people.
labels in process_document =  [0, 0, 1, 0]
labels in getitem [0, 0, 1, 0]
BOS token ID: 50265
7  -  4
doc =  Climate is changing. Polar ice caps are melting. Sports fans are excited. The match was thrilling.
labels in process_document =  [0, 0, 1, 0]
labels in getitem [0, 0, 1, 0]
BOS token ID: 50265
2  -  4
input_ids =  tensor([[    0, 50265,  1234,  ...,     1,     1,     1],
        [    0, 50265,    20,  ...,     1,     1,     1],
     